# 01. Roles Processing

This notebook prepares the node-level metadata used throughout the project.

It reads the raw role table, extracts family information from the `Role` field, derives an `arrested` disruption flag from `Request`, validates the transformation logic, and exports the processed file to `data/processed/roles.csv`.

## Inputs and Outputs

- Input: `../data/raw/Montagna_Roles.csv`
- Output: `../data/processed/roles.csv`

Derived columns:

- `family_role`: role or rank extracted from the raw `Role` text
- `family`: family name extracted from quoted text when present
- `arrested`: `1` for `arrested`, ` arrested`, `house arrest`, or `in jail`; otherwise `0`

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
INPUT_PATH = Path('../data/raw/Montagna_Roles.csv')
OUTPUT_PATH = Path('../data/processed/roles.csv')

roles = pd.read_csv(INPUT_PATH)
roles.head()

## Family Parsing

The project uses a simple rule-based parser aligned with the project notes:

- if the word `family` appears, everything before it becomes `family_role`
- if quotes appear, the content inside the quotes becomes `family`
- if neither appears, keep the original role text and mark the family as `Unknown`
- if the source role is missing, keep both derived fields missing

In [ ]:
role_text = roles['Role'].astype('string').str.strip()

roles['family_role'] = role_text.str.extract(r'^(.*?)\s*family\b', expand=False)
roles['family_role'] = roles['family_role'].fillna(role_text)
roles['family_role'] = roles['family_role'].str.strip().replace('', pd.NA)

family_from_quotes = role_text.str.extract(r'"([^"]+)"|\'\'([^\']+)\'\'', expand=True)
roles['family'] = family_from_quotes.bfill(axis=1).iloc[:, 0]
roles['family'] = roles['family'].fillna('Unknown')

missing_role_mask = role_text.isna() | role_text.eq('')
roles.loc[missing_role_mask, ['family_role', 'family']] = pd.NA

roles[['Node', 'Role', 'family_role', 'family']].head(20)

## Validation of Role and Family Extraction

These checks make the notebook safer to rerun after future edits.

In [ ]:
test_cases = {
    "boss family ''Caltagirone''": ('boss', 'Caltagirone'),
    'boss family "Caltagirone"': ('boss', 'Caltagirone'),
    "member of family ''San Mauro Castelverde''": ('member of', 'San Mauro Castelverde'),
    "Co-founder of family ''Batanesi''": ('Co-founder of', 'Batanesi'),
    'cooperating witness': ('cooperating witness', 'Unknown'),
    'executive family': ('executive', 'Unknown'),
}

for raw_role, expected in test_cases.items():
    sample = pd.Series([raw_role], dtype='string').str.strip()
    parsed_role = sample.str.extract(r'^(.*?)\s*family\b', expand=False).fillna(sample).iloc[0]
    family_match = sample.str.extract(r'"([^"]+)"|\'\'([^\']+)\'\'', expand=True)
    parsed_family = family_match.bfill(axis=1).iloc[0, 0]
    parsed_family = parsed_family if pd.notna(parsed_family) else 'Unknown'
    assert (parsed_role, parsed_family) == expected

roles.loc[[0, 2, 3, 11, 12, 17, 22, 25], ['Node', 'Role', 'family_role', 'family']]

In [ ]:
roles[['family_role', 'family']].value_counts(dropna=False).head(20)

## Arrest Flag for Disruption Analysis

This binary field identifies actors already removed or constrained by law enforcement, which is the basis for the post-disruption network in the main analysis notebook.

In [ ]:
request_text = roles['Request'].astype('string').str.strip().str.lower()
arrest_values = {'house arrest', 'arrested', 'in jail'}
roles['arrested'] = request_text.isin(arrest_values).astype(int)

roles[['Node', 'Request', 'arrested']].head(20)

In [ ]:
assert roles.loc[roles['Request'].eq('arrested'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq(' arrested'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('in jail'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('house arrest'), 'arrested'].eq(1).all()
assert roles.loc[roles['Request'].eq('arrest request denied'), 'arrested'].eq(0).all()

roles['arrested'].value_counts(dropna=False)

## Export Processed Metadata

The processed output will be used by the full network notebook and any downstream reports.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
roles.to_csv(OUTPUT_PATH, index=False)

pd.DataFrame(
    {
        'rows': [len(roles)],
        'unique_nodes': [roles['Node'].nunique()],
        'arrested_nodes': [int(roles['arrested'].sum())],
        'known_families': [int(roles['family'].fillna('Unknown').ne('Unknown').sum())],
        'output_path': [str(OUTPUT_PATH)],
    }
)